
# Geologic Rules → Code Whiteboard (Fluvial)

Purpose: Show, with maximum transparency, how each geologic rule is embodied in the fluvial generators. Each section links the rule, the code anchor, and a short explanation of how the code satisfies or approximates the rule. Use this as a lecture/whiteboard: skim the markdown for rationale, open the linked source if you need to inspect the exact implementation, and tweak parameters live in the code cells.

Scope: fluvial environments currently implemented (meandering, braided, anastomosing, stacked packages, sedimentary overlays). Aeolian/estuarine are outside Phase 1 and are noted as placeholders.


In [ ]:

import inspect
from analog_image_generator import geologic_generators as gg
from analog_image_generator import stacked_channels as sc
from analog_image_generator import stats



### Meandering — rule ↔ code

- Rule: Sinuous belt from control points → **`gg.meander_centerline`** (anchor-fluvial-meander-centerline)
  - Code: draws control points, adds drift, applies sinusoidal modulation → yields centerline array.
- Rule: Variable bankfull width → **`gg.meander_variable_channel`**
  - Code: interpolates width profile (min→max) with noise, builds boolean mask around centerline.
- Rule: Levees as rim via dilation → **`gg.add_levees`**
  - Code: grey dilation + gaussian blur; difference to channel becomes levee rim.
- Rule: Scroll-bar banding → **`gg.add_scroll_bars`**
  - Code: cosine of distance-to-channel to produce bar/scroll rhythm.
- Rule: Oxbow scars at neck cutoffs → **`gg.add_oxbow`**
  - Code: random circles near centerline with probability = `oxbow_probability`.
- Rule: Facies/grayscale composition → **`gg.compose_meandering`** + **`gg.apply_sedimentary_overlays`**
  - Overlays: cross-bedding, ripple, lateral accretion, fining-upward, mudstone.

Key params to tweak:
- `n_control_points`, `amplitude_range`, `drift_fraction`
- `channel_width_min/max`, `scroll_lambda_px`, `oxbow_probability`
- Overlays: `cross_bed_strength`, `ripple_strength`, `ripple_wavelength_px`, `fining_strength`, `mudstone_strength`, `lateral_accretion_strength`


In [ ]:
inspect.getsource(gg.meander_centerline)


### Braided — rule ↔ code

- Rule: Multi-threads across belt → **`gg.braided_threads`**
  - Code: multiple sinuous centerlines with random amp/freq; clipped widths; combined mask.
- Rule: Bar spacing ≈ 4–5× mean width → **`gg.seed_bars`**
  - Code: ellipse patches along threads with spacing tied to `bar_spacing_factor`.
- Rule: Chutes cross-cut bars → **`gg.add_chutes`**
  - Code: line strips connecting threads; frequency = `chute_frequency`.
- Rule: Facies/grayscale composition → **`gg.compose_braided`** + overlays as above.

Key params: `thread_count`, `mean_thread_width`, `bar_spacing_factor`, `chute_frequency`.


In [ ]:
inspect.getsource(gg.braided_threads)


### Anastomosing — rule ↔ code

- Rule: Narrow, stable branches → **`gg.anasto_paths`**
- Rule: Levees for narrow channels → **`gg.add_levees_narrow`** with width/height scale.
- Rule: Wetlands from distance/base quantile → **`gg.make_marsh`**
- Rule: Fans at levee breaches → **`gg.seed_fans`**
- Composition + overlays via **`gg.compose_anasto`** and `apply_sedimentary_overlays`.

Key params: `branch_count`, `levee_width_px`, `levee_height_scale`, `marsh_fraction`, `fan_length_px`.


In [ ]:
inspect.getsource(gg.anasto_paths)


### Stacked packages — rule ↔ code

- Rule: Toggle single vs stacked → **`gg.generate_fluvial`** (mode switch) → **`sc.build_stacked_fluvial`**
- Rule: Ordered packages + styles → **`sc.sequence_packages`** using `package_styles`, `package_thickness_px`, `package_relief_px`, `package_erosion_depth_px`.
- Rule: Relief/erosion trimming → **`sc.apply_relief_slice`** / **`sc.cut_erosional_surface`**.
- Metadata: package_id map + relief/thickness totals in `realization_metadata`.

Key params: `package_count`, `package_mix`, `package_relief_px`, `package_erosion_depth_px`, `package_thickness_px`, `stack_seed`.


In [ ]:
inspect.getsource(sc.build_stacked_fluvial)


### Sedimentary overlays — rule ↔ code

- Rule: Channel-fill sandstone → **`gg.channel_fill_sandstone`**
- Rule: Cross-bedding (trough/planar) → **`gg.apply_cross_bedding`**
- Rule: Ripple marks → **`gg.ripple_mark_texture`** (wavelength tunable)
- Rule: Lateral accretion → **`gg.lateral_accretion_surface`**
- Rule: Fining-upward + mudstone → **`gg.fining_upward_and_mudstone`**
- Rule: Petrology metadata → **`gg._petrology_metadata`** stored in `realization_metadata`.

All controlled via overlay strengths in params: `cross_bed_strength`, `ripple_strength`, `ripple_wavelength_px`, `fining_strength`, `mudstone_strength`, `lateral_accretion_strength`.


In [ ]:
inspect.getsource(gg.apply_sedimentary_overlays)


### Metrics — variogram/β/H/fractal

- Rule: Variogram/β for texture → **`stats.compute_variogram`**, **`stats.fit_power_law`**, **`stats.fractal_dimension`**
- Rule: PSD anisotropy, topology QA → **`stats.psd_anisotropy`**, **`stats.topology_metrics`**
- Rule: Combined metrics payload → **`stats.compute_metrics`** (Phase 1/2) and **`stats.preview_metrics`** (interactive previews)

Preview plots in the interactive notebook come from the same functions; the log-log plot uses `compute_variogram` + `fit_power_law` for β/D/H.


In [ ]:
inspect.getsource(stats.compute_metrics)


### Aeolian / Estuarine

- Generators are placeholders (see `generate_aeolian` / `generate_estuarine`), to be implemented in their respective milestones. We will mirror this rule↔code notebook pattern when those environments land.
